# SPIQ Biomarker Workflow

End-to-end SPIQ initialization on a biomarker feature-selection PCBO instance, followed by two point-selection strategies for multi-start optimization.

In [ ]:
import warnings

import numpy as np
from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp

warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from biomarker_data import pcbo_utils
from biomarker_data.biomarker_utils import convert_pubo_to_ising
from biomarker_data.paths import sample_data_dir
from spiq.qaoa import QAOASolver
from spiq.selection import fixed_interval_selection, k_gaps_selection


In [ ]:
N_QUBITS = 8
SELECT_N_FEATURES = 3
REPS = 2
N_GENS = 4
SEED = 7
NUM_SELECT = 3

np.random.seed(SEED)

data_dir = sample_data_dir(N_QUBITS)
feature_set, feature_to_idx, first_corr, second_corr, third_corr = (
    pcbo_utils.load_features_and_corr_files(str(data_dir))
)

pcbo_obj = pcbo_utils.create_three_body_cubo(
    feature_set,
    first_corr,
    second_corr,
    third_corr,
    feature_to_idx,
    select_n_features=SELECT_N_FEATURES,
)
pubo = {key: float(value) for key, value in pcbo_obj.to_pubo().items()}
cost_hamiltonian = SparsePauliOp.from_list(convert_pubo_to_ising(pubo, N_QUBITS))

circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=REPS)
solver = QAOASolver(cost_hamiltonian, circuit, sim_device="CPU")
solver.err = None
solver.prepare_circuit()
exact_energy = solver.evaluate_exact_energy()

print(f"features={N_QUBITS}, select={SELECT_N_FEATURES}, reps={REPS}")
print(f"exact energy={exact_energy:.6f}")

In [ ]:
solver.run_spiq(n_gens=N_GENS)

best_params = solver.best_spiq_gen_params[::-1]
best_fitness = solver.best_spiq_gen_fitness[::-1]
print(f"SPIQ best energy={solver.energy_best:.6f}")
print(f"candidate points={len(best_fitness)}")

In [ ]:
spaced_params, spaced_fitness = fixed_interval_selection(
    best_params, best_fitness, num_select=NUM_SELECT
)
for i, (params, energy) in enumerate(zip(spaced_params, spaced_fitness)):
    print(f"fixed interval {i + 1}: energy={energy:.6f}, params={params}")

In [ ]:
kgaps_result = k_gaps_selection(
    best_params,
    best_fitness,
    solver,
    num_select=NUM_SELECT,
    rng=np.random.default_rng(SEED),
)
if kgaps_result is None:
    print("k-gaps selection returned no points")
else:
    cluster_params, cluster_fitness, cluster_grads = kgaps_result
    for i, (params, energy, grad) in enumerate(zip(cluster_params, cluster_fitness, cluster_grads)):
        print(f"k-gaps {i + 1}: energy={energy:.6f}, grad_norm={grad:.6f}, params={params}")